**PS1 — Device Failure Prediction (7-day Binary)**

---
SageMaker Studio | MLflow | Feature Store | Model Registry | Model Monitor

Architecture : CUBIC MARS AWS v6
  Gold S3 (Delta) → Feature Store (offline+online) → Train 5 models + stacking
  → MLflow Experiment → MLflow Model Registry → SageMaker Endpoint
  → SageMaker Model Monitor (data quality + model quality)

Target  : will_fail_7d  (0=no failure within 7 days, 1=failure within 7 days)
Grain   : one row per DEVICE_ID per day
Run in  : SageMaker Studio JupyterLab  (kernel: Python 3 — Data Science 3.0)

---

In [ ]:
# ── CELL 1 : Install dependencies ─────────────────────────────────────────────
import subprocess, sys
pkgs = [
    "s3fs", "pyarrow",
    "scikit-learn>=1.4",
    "xgboost>=2.0", "lightgbm>=4.3", "catboost>=1.2",
    "optuna>=3.6", "shap>=0.45",
    "mlflow>=2.13", "sagemaker-mlflow",
    "boto3", "sagemaker>=2.220,<3.0",
    "pandas>=2.0", "matplotlib", "seaborn",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs)
print("All packages installed.")

In [ ]:
# ── CELL 2 : Imports ──────────────────────────────────────────────────────────
import os, json, time, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap, optuna, boto3, sagemaker, joblib, tarfile

# deltalake removed: tables exported as Parquet (DeltaProtocolError on deletionVectors)
from sklearn.model_selection         import TimeSeriesSplit
from sklearn.preprocessing           import StandardScaler, LabelEncoder
from sklearn.pipeline                import Pipeline
from sklearn.linear_model            import LogisticRegression
from sklearn.ensemble                import (RandomForestClassifier,
                                             StackingClassifier)
from sklearn.calibration             import CalibratedClassifierCV
from sklearn.metrics                 import (roc_auc_score, roc_curve,
                                             f1_score, precision_score, recall_score,
                                             average_precision_score, precision_recall_curve,
                                             confusion_matrix, classification_report)
from xgboost   import XGBClassifier
from lightgbm  import LGBMClassifier, early_stopping, log_evaluation
from catboost  import CatBoostClassifier

import mlflow, mlflow.sklearn, mlflow.xgboost
from mlflow.models.signature         import infer_signature

try:
    from sagemaker import Session
except ImportError:
    from sagemaker.session import Session
try:
    from sagemaker import get_execution_role
except ImportError:
    from sagemaker.session import get_execution_role
from sagemaker.feature_store.feature_group      import FeatureGroup
from sagemaker.feature_store.feature_definition import (FeatureDefinition,
                                                        FeatureTypeEnum)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", 60)
print("Imports OK.")

In [ ]:
# ── CELL 3 : Configuration ────────────────────────────────────────────────────
# Fill in your artifact bucket name (the dedicated artefacts bucket)
ARTIFACT_BUCKET = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"

GOLD_BUCKET = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_PREFIX = "chicago/gold/device_ps1_daily"
GOLD_S3     = f"s3://{GOLD_BUCKET}/{GOLD_PREFIX}"

SM_SESSION  = Session()
ROLE        = get_execution_role()
REGION      = boto3.Session().region_name
SM_CLIENT   = boto3.client("sagemaker", region_name=REGION)
S3_CLIENT   = boto3.client("s3",        region_name=REGION)

# ── MLflow — SageMaker Managed Tracking Server
# Create once via AWS console:
#   SageMaker → MLflow → Create tracking server
#   Name: cubic-mars-mlflow-server-dev
#   Artifact store: s3://ARTIFACT_BUCKET/mlflow
#   Size: Small
# Then get the URL:
MLFLOW_SERVER_ARN = "arn:aws:sagemaker:us-east-1:170202974600:mlflow-tracking-server/cubic-mars-mlflow-server-dev"
os.environ["MLFLOW_TRACKING_SERVER_ARN"] = MLFLOW_SERVER_ARN

try:
    MLFLOW_TRACKING_URI = SM_CLIENT.describe_mlflow_tracking_server(
        TrackingServerName="cubic-mars-mlflow-server-dev"
    )["TrackingServerUrl"]
    print(f"MLflow managed server: {MLFLOW_TRACKING_URI}")
except Exception:
    MLFLOW_TRACKING_URI = "file:///home/sagemaker-user/mlruns"
    print("MLflow managed server not found — using local file fallback (EFS)")

mlflow.set_tracking_uri(MLFLOW_SERVER_ARN)  # sagemaker-mlflow resolves arn: scheme
EXPERIMENT_NAME     = "chicago-ps1-device-failure"
MLFLOW_MODEL_NAME   = "ps1-device-failure"
MODEL_REGISTRY_NAME = "chicago-ps1-device-failure"   # SageMaker Model Package Group
FS_GROUP_NAME       = "chicago-ps1-features"
ENDPOINT_NAME       = "chicago-ps1-failure-v1"

mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Region      : {REGION}")
print(f"Gold S3     : {GOLD_S3}")
print(f"Artifacts   : s3://{ARTIFACT_BUCKET}/")
print(f"MLflow URI  : {MLFLOW_TRACKING_URI}")

In [ ]:
# ── CELL 4 : Load Gold table (Parquet on S3) ──────────────────────────────────
# G01 source: mars_dev.gold.device_ps1_daily exported via notebooks/export_gold_to_s3.py
# Column note: gold SQL writes "transit_day"; aliased to "event_date" below.

print(f"Loading Gold table from: {GOLD_S3}")

df = pd.read_parquet(GOLD_S3, storage_options={"client_kwargs": {"region_name": REGION}})
GOLD_VERSION = None   # Parquet has no version; use None for MLflow tags
DATA_SOURCE  = "parquet"

# gold SQL writes transit_day; alias to event_date used throughout notebook
if "transit_day" in df.columns and "event_date" not in df.columns:
    df = df.rename(columns={"transit_day": "event_date"})
df["event_date"] = pd.to_datetime(df["event_date"])

print(f"Parquet source  : confirmed")
print(f"Shape           : {df.shape}")
print(f"Date range      : {df['event_date'].min().date()}  →  {df['event_date'].max().date()}")
print(f"\nTarget distribution:\n{df['will_fail_7d'].value_counts()}")
print(f"Failure rate    : {df['will_fail_7d'].mean():.2%}")
print(f"\nDevice mix:\n{df.drop_duplicates('DEVICE_ID')['mars_device_category'].value_counts()}")

In [ ]:
# ── CELL 5 : EDA ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df["will_fail_7d"].value_counts().plot(
    kind="bar", ax=axes[0], color=["#43A047","#E53935"], edgecolor="white")
axes[0].set_title("Target: will_fail_7d")

(1 - df.isnull().mean()).sort_values().head(20).plot(
    kind="barh", ax=axes[1], color="#1565C0")
axes[1].set_title("Feature completeness (lowest 20)")

df.groupby("event_date")["will_fail_7d"].sum().plot(
    ax=axes[2], color="#E53935", linewidth=0.8)
axes[2].set_title("Daily failure count over time")

plt.tight_layout()
plt.savefig("/tmp/ps1_eda.png", dpi=100)
plt.show()
print("EDA saved → /tmp/ps1_eda.png")

In [ ]:
# ── CELL 6 : Feature engineering & temporal splits ───────────────────────────
TARGET   = "will_fail_7d"
ID_COLS  = ["DEVICE_ID", "event_date", "split"]

# Derived rolling features (do NOT put in Gold — keep Gold schema stable)
for col, window in [("oos_flag", 7), ("oos_flag", 30), ("daily_event_count", 7)]:
    if col in df.columns:
        df[f"{col}_roll{window}"] = (
            df.groupby("DEVICE_ID")[col]
              .transform(lambda x: x.rolling(window, min_periods=1).sum()))

if "daily_event_count" in df.columns:
    df["event_rate_zscore"] = (
        df.groupby("DEVICE_ID")["daily_event_count"]
          .transform(lambda x: (x - x.mean()) / (x.std() + 1e-6)))

if "device_type" in df.columns:
    df["device_type_enc"] = LabelEncoder().fit_transform(
        df["device_type"].fillna("UNKNOWN"))

# ── S21 use_revenue_daily additions (2026-06-25) ─────────────────────────────
# use_revenue_decline_flag arrives as BOOLEAN from Delta/Spark; cast to int
# so it enters FEATURE_COLS (is_numeric_dtype returns False for bool)
if "use_revenue_decline_flag" in df.columns:
    df["use_revenue_decline_flag"] = df["use_revenue_decline_flag"].astype(int)

# 7-day rolling mean of daily revenue (additional temporal feature for PS1)
if "use_revenue_cents" in df.columns:
    df["use_revenue_cents_roll7"] = (
        df.sort_values(["DEVICE_ID", "event_date"])
          .groupby("DEVICE_ID")["use_revenue_cents"]
          .transform(lambda x: x.rolling(7, min_periods=1).mean()))

print(f"use_revenue columns in gold: {[c for c in df.columns if c.startswith('use_revenue') or c.startswith('use_txn')]}")

# Feature columns = numeric columns excluding IDs + target
FEATURE_COLS = [
    c for c in df.columns
    if c not in ID_COLS + [TARGET]
    and pd.api.types.is_numeric_dtype(df[c])
]
print(f"Features : {len(FEATURE_COLS)}")

# Fill nulls — XGBoost/LightGBM/CatBoost handle NaN natively;
# fill needed only for sklearn models (LR, RF)
df[FEATURE_COLS] = df[FEATURE_COLS].fillna(df[FEATURE_COLS].median())

# Temporal split — NEVER random on time-series
# Train      : 2024-01-01 → 2025-09-30
# Validation : 2025-10-01 → 2025-12-31
# Test       : 2026-01-01 → 2026-04-30
train_df = df[df["event_date"] <  "2025-10-01"]
val_df   = df[(df["event_date"] >= "2025-10-01") & (df["event_date"] < "2026-01-01")]
test_df  = df[df["event_date"] >= "2026-01-01"]

X_train, y_train = train_df[FEATURE_COLS].values, train_df[TARGET].values
X_val,   y_val   = val_df[FEATURE_COLS].values,   val_df[TARGET].values
X_test,  y_test  = test_df[FEATURE_COLS].values,  test_df[TARGET].values
X_cv             = np.vstack([X_train, X_val])
y_cv             = np.concatenate([y_train, y_val])

neg, pos         = (y_train == 0).sum(), (y_train == 1).sum()
SCALE_POS_WEIGHT = round(neg / pos, 2)

# ── Split summary ─────────────────────────────────────────────────────────────
print(f"\n── Temporal Split Summary ─────────────────────────────────────────────")
print(f"  {'Split':<8}  {'Rows':>9}  {'Date range':<31}  {'No Fail':>10}  {'Fail':>8}  {'Fail%':>7}  {'Ratio':>7}")
for sname, sdf, sy in [("Train", train_df, y_train),
                        ("Val",   val_df,   y_val),
                        ("Test",  test_df,  y_test)]:
    n0, n1 = (sy == 0).sum(), (sy == 1).sum()
    dr = f"{sdf['event_date'].min().date()} → {sdf['event_date'].max().date()}"
    print(f"  {sname:<8}  {len(sdf):>9,}  {dr:<31}  {n0:>10,}  {n1:>8,}  {n1/len(sy):>7.2%}  {n0/n1:>5.1f}:1")

# ── Class imbalance handling ──────────────────────────────────────────────────
print(f"\n── Class Imbalance Handling ───────────────────────────────────────────")
print(f"  Imbalance  : {neg:,} No Fail  vs  {pos:,} Fail  ({SCALE_POS_WEIGHT:.1f}:1 in train)")
print(f"  LR         : class_weight='balanced'           → sklearn upweights Fail class inversely")
print(f"  RF         : class_weight='balanced_subsample' → per-tree class resampling")
print(f"  XGB / LGB  : scale_pos_weight={SCALE_POS_WEIGHT}             → loss gradient amplified for Fail")
print(f"  CatBoost   : scale_pos_weight={SCALE_POS_WEIGHT}             → same mechanism as XGB/LGB")
print(f"  Stacking   : inherits from all 5 base estimators above")
print(f"  SMOTE      : NOT used — synthetic oversampling breaks temporal ordering")

# ── MLflow rerun behaviour ────────────────────────────────────────────────────
print(f"\n── MLflow Rerun Behaviour ─────────────────────────────────────────────")
print(f"  Each rerun creates NEW runs (fresh run_id) under the same experiment.")
print(f"  Old runs are NEVER overwritten — duplicates accumulate per rerun.")
print(f"  CELL 20 re-ranks all runs by test_auc every time → picks the current best.")
print(f"  To clean up: MLflow UI → '{EXPERIMENT_NAME}' → select old runs → Delete.")
print(f"  Tip: add mlflow.set_tag('run_date','YYYY-MM-DD') before each run to track versions.")

In [ ]:
# ── CELL 7 : SageMaker Feature Store — create Feature Group ──────────────────
# Offline store → S3 (training, batch inference)
# Online store  → DynamoDB (<10 ms, real-time inference feature lookup)

fs_df = train_df[FEATURE_COLS + [TARGET, "DEVICE_ID", "event_date"]].copy()
fs_df["event_time"] = fs_df["event_date"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
fs_df["DEVICE_ID"]  = fs_df["DEVICE_ID"].astype(str)
fs_df               = fs_df.drop(columns=["event_date"])

def dtype_to_fs_type(dtype):
    if pd.api.types.is_float_dtype(dtype):   return FeatureTypeEnum.FRACTIONAL
    if pd.api.types.is_integer_dtype(dtype): return FeatureTypeEnum.INTEGRAL
    return FeatureTypeEnum.STRING

feature_definitions = [
    FeatureDefinition(feature_name=col,
                      feature_type=dtype_to_fs_type(fs_df[col].dtype))
    for col in fs_df.columns
]

feature_group = FeatureGroup(
    name                = FS_GROUP_NAME,
    sagemaker_session   = SM_SESSION,
    feature_definitions = feature_definitions,
)

try:
    feature_group.create(
        s3_uri                   = f"s3://{ARTIFACT_BUCKET}/feature-store/{FS_GROUP_NAME}",
        record_identifier_name   = "DEVICE_ID",
        event_time_feature_name  = "event_time",
        role_arn                 = ROLE,
        enable_online_store      = True,
        description              = "PS1 daily device features — Chicago Ventra (CUBIC MARS AWS v6)",
    )
    status = ""
    while status != "Created":
        time.sleep(15)
        status = feature_group.describe().get("FeatureGroupStatus")
        print(f"  Feature Group status: {status}")
    print("Feature Group created.")
except Exception as e:
    print(f"Feature Group already exists or error: {e}")

In [ ]:
# ── CELL 8 : Ingest training features into Feature Store ─────────────────────
print("Ingesting features…")
feature_group.ingest(
    data_frame  = fs_df.reset_index(drop=True),
    max_workers = 4,
    wait        = True,
)
print(f"Ingested {len(fs_df):,} records → {FS_GROUP_NAME}")

# Verify online store
sample_id = fs_df["DEVICE_ID"].iloc[0]
record    = feature_group.get_record(record_identifier_value_as_string=sample_id)
print(f"\nOnline store sample ({sample_id}):")
for f in record[:4]:
    print(f"  {f['FeatureName']}: {f['ValueAsString']}")

In [ ]:
# ── CELL 9 : Helper functions ─────────────────────────────────────────────────

COMMON_TAGS = {
    "ps"           : "PS1",
    "target"       : "will_fail_7d",
    "city"         : "chicago",
    "gold_version" : str(GOLD_VERSION),
    "grain"        : "device_day",
    "architecture" : "CUBIC-MARS-AWS-v6",
}

def evaluate(model, X, y, split="test"):
    prob = model.predict_proba(X)[:, 1]
    pred = (prob >= 0.5).astype(int)
    return {
        f"{split}_auc"  : round(roc_auc_score(y, prob), 4),
        f"{split}_ap"   : round(average_precision_score(y, prob), 4),
        f"{split}_f1"   : round(f1_score(y, pred), 4),
        f"{split}_prec" : round(precision_score(y, pred, zero_division=0), 4),
        f"{split}_rec"  : round(recall_score(y, pred, zero_division=0), 4),
    }

def log_cm(model, X, y, name, split="test"):
    pred = (model.predict_proba(X)[:, 1] >= 0.5).astype(int)
    cm   = confusion_matrix(y, pred)
    fig, ax = plt.subplots(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No Fail","Fail"],
                yticklabels=["No Fail","Fail"])
    ax.set_title(f"{name} — Confusion Matrix ({split})")
    path = f"/tmp/cm_{name}_{split}.png"
    fig.savefig(path, bbox_inches="tight", dpi=80)
    plt.show(); plt.close(fig)
    mlflow.log_artifact(path)

def plot_roc_pr(model, name):
    """ROC + Precision-Recall curves for train / val / test on one figure."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    colors = {"train": "#1565C0", "val": "#E65100", "test": "#2E7D32"}
    for label, X, y in [("train", X_train, y_train),
                         ("val",   X_val,   y_val),
                         ("test",  X_test,  y_test)]:
        prob        = model.predict_proba(X)[:, 1]
        fpr, tpr, _ = roc_curve(y, prob)
        pr, rc, _   = precision_recall_curve(y, prob)
        auc_val     = roc_auc_score(y, prob)
        ap_val      = average_precision_score(y, prob)
        col         = colors[label]
        axes[0].plot(fpr, tpr, color=col, lw=1.6,
                     label=f"{label}  AUC={auc_val:.4f}")
        axes[1].plot(rc,  pr,  color=col, lw=1.6,
                     label=f"{label}  AP={ap_val:.4f}")
    axes[0].plot([0,1],[0,1],"k--", lw=0.7)
    axes[0].set(xlabel="False Positive Rate", ylabel="True Positive Rate",
                title=f"{name} — ROC Curve")
    axes[0].legend(fontsize=9)
    axes[1].set(xlabel="Recall", ylabel="Precision",
                title=f"{name} — Precision-Recall Curve")
    axes[1].legend(fontsize=9)
    fig.suptitle(f"{name}  |  Train/Val/Test", fontsize=11, y=1.01)
    fig.tight_layout()
    path = f"/tmp/roc_pr_{name}.png"
    fig.savefig(path, dpi=100, bbox_inches="tight")
    plt.show(); plt.close(fig)
    mlflow.log_artifact(path)

def log_feat_importance(importances, name, top_n=20):
    """Horizontal bar chart for top-N features; chart logged to MLflow."""
    fi = pd.Series(importances, index=FEATURE_COLS).nlargest(top_n)
    fig, ax = plt.subplots(figsize=(9, 5))
    fi.sort_values().plot(kind="barh", ax=ax, color="#1565C0", edgecolor="white")
    ax.set_title(f"{name} — Top {top_n} Feature Importances")
    ax.set_xlabel("Importance")
    fig.tight_layout()
    path = f"/tmp/fi_{name}.png"
    fig.savefig(path, dpi=100)
    plt.show(); plt.close(fig)
    mlflow.log_artifact(path)
    print(f"\n── {name} — Top 10 Feature Importances ──────────────────────────")
    print(fi.head(10).to_string())

def log_clf_report(model, name):
    """Classification report for train / val / test; test report saved to MLflow."""
    for split, X, y in [("train", X_train, y_train),
                         ("val",   X_val,   y_val),
                         ("test",  X_test,  y_test)]:
        pred = (model.predict_proba(X)[:, 1] >= 0.5).astype(int)
        print(f"\n── {name} [{split}] ─────────────────────────────────────────")
        print(classification_report(y, pred,
                                    target_names=["No Fail", "Fail"], digits=4))
    pred_test = (model.predict_proba(X_test)[:, 1] >= 0.5).astype(int)
    rep       = classification_report(y_test, pred_test,
                                      target_names=["No Fail","Fail"], digits=4)
    path = f"/tmp/report_{name}.txt"
    with open(path, "w") as f:
        f.write(rep)
    mlflow.log_artifact(path)

In [ ]:
# ── CELL 10 : MODEL 1 — Logistic Regression (baseline) ───────────────────────
# Interpretable linear baseline. Needs StandardScaler.
# Sets the minimum AUC bar all other models must beat.

with mlflow.start_run(run_name="logistic_regression") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "linear_baseline"})

    lr = Pipeline([
        ("sc",  StandardScaler()),
        ("clf", LogisticRegression(C=0.1, class_weight="balanced",
                                   max_iter=1000, solver="lbfgs", random_state=42)),
    ])
    lr.fit(X_train, y_train)

    vm = evaluate(lr, X_val,  y_val,  "val")
    tm = evaluate(lr, X_test, y_test, "test")
    mlflow.log_params({"C":0.1,"class_weight":"balanced","scaler":"standard"})
    mlflow.log_metrics({**vm, **tm})
    log_cm(lr, X_test, y_test, "lr")
    plot_roc_pr(lr, "lr")
    # LR: |coefficient| = feature importance (after standardisation all features are on same scale)
    log_feat_importance(np.abs(lr.named_steps["clf"].coef_[0]), "lr")
    log_clf_report(lr, "lr")

    sig = infer_signature(X_train, lr.predict_proba(X_train))
    mlflow.sklearn.log_model(lr, "model", signature=sig,
                             input_example=X_train[:3])
    LR_RUN = run.info.run_id

print(f"\n── LR Metrics ─────────────────────────────────────────────────────────")
print(f"  val  → AUC={vm['val_auc']}  AP={vm['val_ap']}  F1={vm['val_f1']}  Prec={vm['val_prec']}  Rec={vm['val_rec']}")
print(f"  test → AUC={tm['test_auc']}  AP={tm['test_ap']}  F1={tm['test_f1']}  Prec={tm['test_prec']}  Rec={tm['test_rec']}")

In [ ]:
# ── CELL 11 : MODEL 2 — Random Forest ────────────────────────────────────────
# Bagging ensemble. No scaling. Handles non-linearities.
# balanced_subsample = re-samples class weights per tree.

with mlflow.start_run(run_name="random_forest") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "bagging_ensemble"})

    rf = RandomForestClassifier(
        n_estimators     = 300,
        max_depth        = 12,
        min_samples_leaf = 20,
        class_weight     = "balanced_subsample",
        n_jobs           = -1,
        random_state     = 42,
    )
    rf.fit(X_train, y_train)

    vm = evaluate(rf, X_val,  y_val,  "val")
    tm = evaluate(rf, X_test, y_test, "test")
    mlflow.log_params({"n_estimators":300,"max_depth":12,
                       "class_weight":"balanced_subsample"})
    mlflow.log_metrics({**vm, **tm})
    log_cm(rf, X_test, y_test, "rf")
    plot_roc_pr(rf, "rf")
    log_feat_importance(rf.feature_importances_, "rf")
    log_clf_report(rf, "rf")

    sig = infer_signature(X_train, rf.predict_proba(X_train))
    mlflow.sklearn.log_model(rf, "model", signature=sig,
                             input_example=X_train[:3])
    RF_RUN = run.info.run_id

print(f"\n── RF Metrics ─────────────────────────────────────────────────────────")
print(f"  val  → AUC={vm['val_auc']}  AP={vm['val_ap']}  F1={vm['val_f1']}  Prec={vm['val_prec']}  Rec={vm['val_rec']}")
print(f"  test → AUC={tm['test_auc']}  AP={tm['test_ap']}  F1={tm['test_f1']}  Prec={tm['test_prec']}  Rec={tm['test_rec']}")

In [ ]:
# ── CELL 12 : MODEL 3 — XGBoost ──────────────────────────────────────────────
# Primary boosting model. scale_pos_weight handles class imbalance.
# Native NaN support — no imputation needed.
# early_stopping_rounds on val AUC.

XGB_PARAMS = dict(
    n_estimators          = 600,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 10,
    gamma                 = 1.0,
    reg_alpha             = 0.1,
    reg_lambda            = 1.0,
    scale_pos_weight      = SCALE_POS_WEIGHT,
    eval_metric           = "auc",
    early_stopping_rounds = 40,
    tree_method           = "hist",
    random_state          = 42,
    n_jobs                = -1,
)

with mlflow.start_run(run_name="xgboost") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "gradient_boosting_xgb"})

    xgb = XGBClassifier(**XGB_PARAMS)
    xgb.fit(X_train, y_train,
            eval_set=[(X_val, y_val)], verbose=100)

    vm = evaluate(xgb, X_val,  y_val,  "val")
    tm = evaluate(xgb, X_test, y_test, "test")
    mlflow.log_params({**XGB_PARAMS, "best_iter": xgb.best_iteration})
    mlflow.log_metrics({**vm, **tm})
    log_cm(xgb, X_test, y_test, "xgb")
    plot_roc_pr(xgb, "xgb")
    log_feat_importance(xgb.feature_importances_, "xgb")
    log_clf_report(xgb, "xgb")

    sig = infer_signature(
        pd.DataFrame(X_train, columns=FEATURE_COLS),
        xgb.predict_proba(X_train))
    # xgboost flavor bypasses skops UntrustedTypesFoundException
    mlflow.xgboost.log_model(xgb, "model", signature=sig,
                              input_example=pd.DataFrame(X_train[:3], columns=FEATURE_COLS))
    XGB_RUN = run.info.run_id

print(f"\n── XGB Metrics ────────────────────────────────────────────────────────")
print(f"  val  → AUC={vm['val_auc']}  AP={vm['val_ap']}  F1={vm['val_f1']}  Prec={vm['val_prec']}  Rec={vm['val_rec']}")
print(f"  test → AUC={tm['test_auc']}  AP={tm['test_ap']}  F1={tm['test_f1']}  Prec={tm['test_prec']}  Rec={tm['test_rec']}")
print(f"  best_iter={xgb.best_iteration}")

In [ ]:
# ── CELL 13 : MODEL 4 — LightGBM ─────────────────────────────────────────────
# Faster than XGBoost on wide datasets. Leaf-wise growth.
# early_stopping via callback (LightGBM >=4 API).

LGB_PARAMS = dict(
    n_estimators     = 600,
    num_leaves       = 63,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_samples= 20,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    scale_pos_weight = SCALE_POS_WEIGHT,
    n_jobs           = -1,
    random_state     = 42,
    verbose          = -1,
)

with mlflow.start_run(run_name="lightgbm") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "gradient_boosting_lgbm"})

    lgb = LGBMClassifier(**LGB_PARAMS)
    lgb.fit(X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[early_stopping(40, verbose=False),
                       log_evaluation(100)])

    vm = evaluate(lgb, X_val,  y_val,  "val")
    tm = evaluate(lgb, X_test, y_test, "test")
    mlflow.log_params({**LGB_PARAMS, "best_iter": lgb.best_iteration_})
    mlflow.log_metrics({**vm, **tm})
    log_cm(lgb, X_test, y_test, "lgb")
    plot_roc_pr(lgb, "lgb")
    log_feat_importance(lgb.feature_importances_, "lgb")
    log_clf_report(lgb, "lgb")

    sig = infer_signature(X_train, lgb.predict_proba(X_train))
    # pickle serialization avoids skops UntrustedTypesFoundException for LightGBM
    mlflow.sklearn.log_model(lgb, "model", signature=sig,
                             input_example=X_train[:3],
                             serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE)
    LGB_RUN = run.info.run_id

print(f"\n── LGB Metrics ────────────────────────────────────────────────────────")
print(f"  val  → AUC={vm['val_auc']}  AP={vm['val_ap']}  F1={vm['val_f1']}  Prec={vm['val_prec']}  Rec={vm['val_rec']}")
print(f"  test → AUC={tm['test_auc']}  AP={tm['test_ap']}  F1={tm['test_f1']}  Prec={tm['test_prec']}  Rec={tm['test_rec']}")
print(f"  best_iter={lgb.best_iteration_}")

In [ ]:
# ── CELL 14 : MODEL 5 — CatBoost ─────────────────────────────────────────────
# Best on categorical features without manual encoding.
# scale_pos_weight handles class imbalance.

cat_idx = [i for i, c in enumerate(FEATURE_COLS)
           if any(k in c for k in ("device_type","facility","subsystem","enc"))]

CB_PARAMS = dict(
    iterations            = 600,
    depth                 = 7,
    learning_rate         = 0.05,
    l2_leaf_reg           = 3,
    scale_pos_weight      = SCALE_POS_WEIGHT,
    eval_metric           = "AUC",
    early_stopping_rounds = 40,
    random_seed           = 42,
    task_type             = "CPU",
    verbose               = 100,
)

with mlflow.start_run(run_name="catboost") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "gradient_boosting_catboost"})

    cb = CatBoostClassifier(**CB_PARAMS)
    cb.fit(X_train, y_train,
           eval_set=(X_val, y_val),
           cat_features=cat_idx if cat_idx else None)

    vm = evaluate(cb, X_val,  y_val,  "val")
    tm = evaluate(cb, X_test, y_test, "test")
    mlflow.log_params({**{k:v for k,v in CB_PARAMS.items() if k!="verbose"},
                       "best_iter": cb.best_iteration_,
                       "cat_features_count": len(cat_idx)})
    mlflow.log_metrics({**vm, **tm})
    log_cm(cb, X_test, y_test, "cb")
    plot_roc_pr(cb, "cb")
    log_feat_importance(cb.get_feature_importance(), "cb")
    log_clf_report(cb, "cb")

    cb.save_model("/tmp/ps1_catboost.cbm")
    mlflow.log_artifact("/tmp/ps1_catboost.cbm", "catboost_native")

    sig = infer_signature(X_train, cb.predict_proba(X_train))
    # pickle serialization avoids skops UntrustedTypesFoundException for CatBoost
    mlflow.sklearn.log_model(cb, "model", signature=sig,
                             input_example=X_train[:3],
                             serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE)
    CB_RUN = run.info.run_id

print(f"\n── CB Metrics ─────────────────────────────────────────────────────────")
print(f"  val  → AUC={vm['val_auc']}  AP={vm['val_ap']}  F1={vm['val_f1']}  Prec={vm['val_prec']}  Rec={vm['val_rec']}")
print(f"  test → AUC={tm['test_auc']}  AP={tm['test_ap']}  F1={tm['test_f1']}  Prec={tm['test_prec']}  Rec={tm['test_rec']}")
print(f"  best_iter={cb.best_iteration_}")

In [ ]:
# ── CELL 15 : Compare base models ────────────────────────────────────────────
client = mlflow.tracking.MlflowClient()
exp_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
rows   = []
for r in client.search_runs([exp_id], order_by=["metrics.test_auc DESC"]):
    if r.data.tags.get("mlflow.runName","").startswith("stage"):
        continue
    rows.append({
        "model"    : r.data.tags.get("mlflow.runName",""),
        "run_id"   : r.info.run_id[:8],
        "val_auc"  : r.data.metrics.get("val_auc"),
        "test_auc" : r.data.metrics.get("test_auc"),
        "test_f1"  : r.data.metrics.get("test_f1"),
        "test_ap"  : r.data.metrics.get("test_ap"),
    })

cmp = pd.DataFrame(rows).sort_values("test_auc", ascending=False)
print("\n── Base Model Comparison ──────────────────────────────────────")
print(cmp.to_string(index=False))

BEST_BASE = cmp.iloc[0]
print(f"\nBest base model: {BEST_BASE['model']}  AUC={BEST_BASE['test_auc']}")

In [ ]:
# ── CELL 16 : Optuna hyperparameter tuning (XGBoost) ─────────────────────────
# ALL-FOLD rule: objective averages AUC across every TimeSeriesSplit fold —
# not just the last fold — to prevent temporal overfitting.
# SMOTE is NOT used: synthetic resampling on time-series leaks future signal.

N_TRIALS  = 50          # raise to 200 for production
TS_CV     = TimeSeriesSplit(n_splits=5)

def optuna_objective(trial):
    p = dict(
        n_estimators     = trial.suggest_int  ("n_estimators",    200, 800),
        max_depth        = trial.suggest_int  ("max_depth",         4,   9),
        learning_rate    = trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        subsample        = trial.suggest_float("subsample",      0.5, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree",0.5, 1.0),
        min_child_weight = trial.suggest_int  ("min_child_weight",  1,  30),
        gamma            = trial.suggest_float("gamma",           0.0, 3.0),
        reg_alpha        = trial.suggest_float("reg_alpha",  1e-4, 10., log=True),
        reg_lambda       = trial.suggest_float("reg_lambda", 1e-4, 10., log=True),
        scale_pos_weight = SCALE_POS_WEIGHT,
        tree_method="hist", random_state=42, n_jobs=-1,
    )
    # ALL folds averaged — not last fold only
    scores = []
    for step, (tr, va) in enumerate(TS_CV.split(X_cv)):
        m = XGBClassifier(**p)
        m.fit(X_cv[tr], y_cv[tr],
              eval_set=[(X_cv[va], y_cv[va])], verbose=False)
        scores.append(roc_auc_score(y_cv[va],
                                    m.predict_proba(X_cv[va])[:,1]))
        trial.report(np.mean(scores), step=step)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return np.mean(scores)

study = optuna.create_study(
    direction  = "maximize",
    study_name = "ps1-xgb-optuna",
    pruner     = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1),
)
study.optimize(optuna_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"Best CV AUC (all-fold avg): {study.best_value:.4f}")
BEST_PARAMS = {**study.best_params,
               "scale_pos_weight": SCALE_POS_WEIGHT,
               "tree_method": "hist", "random_state": 42, "n_jobs": -1}

with mlflow.start_run(run_name="xgboost_optuna") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "xgboost_optuna_tuned"})
    mlflow.log_params({**BEST_PARAMS,
                       "optuna_trials": N_TRIALS,
                       "optuna_cv_folds": 5,
                       "optuna_best_cv": round(study.best_value, 4),
                       "smote": False})

    xgb_tuned = XGBClassifier(**BEST_PARAMS)
    xgb_tuned.fit(X_cv, y_cv, verbose=False)

    vm = evaluate(xgb_tuned, X_val,  y_val,  "val")
    tm = evaluate(xgb_tuned, X_test, y_test, "test")
    mlflow.log_metrics({**vm, **tm,
                        "optuna_cv_auc": round(study.best_value, 4)})
    log_cm(xgb_tuned, X_test, y_test, "xgb_optuna")
    plot_roc_pr(xgb_tuned, "xgb_optuna")
    log_feat_importance(xgb_tuned.feature_importances_, "xgb_optuna")
    log_clf_report(xgb_tuned, "xgb_optuna")

    # Optuna charts
    for fname, fig_fn in [
        ("optuna_history.png",       optuna.visualization.matplotlib.plot_optimization_history),
        ("optuna_param_import.png",  optuna.visualization.matplotlib.plot_param_importances),
    ]:
        try:
            fig_fn(study); plt.tight_layout()
            plt.savefig(f"/tmp/{fname}", dpi=80); plt.close()
            mlflow.log_artifact(f"/tmp/{fname}")
        except Exception:
            pass

    sig = infer_signature(
        pd.DataFrame(X_train, columns=FEATURE_COLS),
        xgb_tuned.predict_proba(X_train))
    # xgboost flavor bypasses skops UntrustedTypesFoundException
    mlflow.xgboost.log_model(xgb_tuned, "model", signature=sig,
                              input_example=pd.DataFrame(X_train[:3], columns=FEATURE_COLS))
    XGB_TUNED_RUN = run.info.run_id

print(f"\n── XGB Optuna Metrics ─────────────────────────────────────────────────")
print(f"  optuna_cv_auc={study.best_value:.4f}")
print(f"  val  → AUC={vm['val_auc']}  AP={vm['val_ap']}  F1={vm['val_f1']}  Prec={vm['val_prec']}  Rec={vm['val_rec']}")
print(f"  test → AUC={tm['test_auc']}  AP={tm['test_ap']}  F1={tm['test_f1']}  Prec={tm['test_prec']}  Rec={tm['test_rec']}")

In [ ]:
# ── CELL 17 : Stacking Ensemble ───────────────────────────────────────────────
# 5 base estimators → meta Logistic Regression (isotonic calibration)
# CV: TimeSeriesSplit(5) — temporal order preserved throughout
# passthrough=True: original features passed alongside meta-features to final estimator
# SMOTE NOT used — time-series must not be synthetically resampled
# No feature importance: stacking blends heterogeneous models — no unified importance

with mlflow.start_run(run_name="stacking_ensemble") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "stacking_ensemble"})
    mlflow.log_params({
        "base_models"    : "LR+RF+XGB_tuned+LGB+CatBoost",
        "final_estimator": "LogisticRegression+isotonic",
        "cv_strategy"    : "TimeSeriesSplit(5)",
        "passthrough"    : True,
        "smote"          : False,
    })

    base = [
        ("lr",  Pipeline([("sc", StandardScaler()),
                           ("clf", LogisticRegression(C=0.1, class_weight="balanced",
                                                      max_iter=1000, random_state=42))])),
        ("rf",  RandomForestClassifier(n_estimators=200, max_depth=10,
                                        class_weight="balanced_subsample",
                                        n_jobs=-1, random_state=42)),
        ("xgb", XGBClassifier(**{**BEST_PARAMS, "n_estimators": 300}, verbosity=0)),
        ("lgb", LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                scale_pos_weight=SCALE_POS_WEIGHT,
                                n_jobs=-1, random_state=42, verbose=-1)),
        ("cb",  CatBoostClassifier(iterations=300, learning_rate=0.05,
                                    scale_pos_weight=SCALE_POS_WEIGHT,
                                    random_seed=42, verbose=0)),
    ]
    stack = StackingClassifier(
        estimators     = base,
        final_estimator= CalibratedClassifierCV(
            LogisticRegression(C=1.0, max_iter=500), cv=3, method="isotonic"),
        cv             = TimeSeriesSplit(n_splits=5),
        stack_method   = "predict_proba",
        passthrough    = True,
        n_jobs         = -1,
    )
    stack.fit(X_cv, y_cv)

    vm = evaluate(stack, X_val,  y_val,  "val")
    tm = evaluate(stack, X_test, y_test, "test")
    mlflow.log_metrics({**vm, **tm})
    log_cm(stack, X_test, y_test, "stack")
    plot_roc_pr(stack, "stack")
    log_clf_report(stack, "stack")

    sig = infer_signature(X_train, stack.predict_proba(X_train))
    # pickle serialization: stacking contains XGB/LGB/CatBoost — skops rejects all three
    mlflow.sklearn.log_model(stack, "model", signature=sig,
                             input_example=X_train[:3],
                             serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE)
    STACK_RUN = run.info.run_id

print(f"\n── Stack Metrics ──────────────────────────────────────────────────────")
print(f"  val  → AUC={vm['val_auc']}  AP={vm['val_ap']}  F1={vm['val_f1']}  Prec={vm['val_prec']}  Rec={vm['val_rec']}")
print(f"  test → AUC={tm['test_auc']}  AP={tm['test_ap']}  F1={tm['test_f1']}  Prec={tm['test_prec']}  Rec={tm['test_rec']}")

In [ ]:
# ── CELL 18 : SHAP analysis — TreeExplainer on xgb_tuned ─────────────────────
# plot_title kwarg used to suppress DeprecationWarning in shap >= 0.45

print("Computing SHAP values…")
X_shap = pd.DataFrame(X_test[:2000], columns=FEATURE_COLS)

explainer   = shap.TreeExplainer(xgb_tuned)
shap_values = explainer(X_shap)

with mlflow.start_run(run_name="xgboost_shap") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "shap_analysis",
                     "parent_run": XGB_TUNED_RUN})

    # Bar — mean |SHAP|
    shap.plots.bar(shap_values, max_display=20, show=False)
    fig = plt.gcf(); ax = plt.gca()
    ax.set_title("PS1 XGBoost — Mean |SHAP| Feature Importance", pad=12)
    fig.tight_layout(); fig.savefig("/tmp/shap_bar.png", dpi=100); plt.close(fig)
    mlflow.log_artifact("/tmp/shap_bar.png")

    # Beeswarm
    shap.plots.beeswarm(shap_values, max_display=20, show=False)
    fig = plt.gcf(); ax = plt.gca()
    ax.set_title("PS1 XGBoost — SHAP Impact Distribution", pad=12)
    fig.tight_layout(); fig.savefig("/tmp/shap_beeswarm.png", dpi=100); plt.close(fig)
    mlflow.log_artifact("/tmp/shap_beeswarm.png")

    # Dependence — top feature
    top_feat = pd.Series(np.abs(shap_values.values).mean(0),
                         index=FEATURE_COLS).idxmax()
    shap.plots.scatter(shap_values[:, top_feat], color=shap_values, show=False)
    fig = plt.gcf(); ax = plt.gca()
    ax.set_title(f"SHAP Dependence — {top_feat}", pad=12)
    fig.tight_layout(); fig.savefig("/tmp/shap_dep.png", dpi=100); plt.close(fig)
    mlflow.log_artifact("/tmp/shap_dep.png")

    # SHAP importance CSV
    shap_imp = (pd.Series(np.abs(shap_values.values).mean(0), index=FEATURE_COLS)
                  .sort_values(ascending=False)
                  .reset_index()
                  .rename(columns={"index":"feature", 0:"mean_abs_shap"}))
    shap_imp.to_csv("/tmp/shap_importance.csv", index=False)
    mlflow.log_artifact("/tmp/shap_importance.csv")
    mlflow.log_param("top_shap_feature", top_feat)

print(f"\nTop 5 SHAP features:\n{shap_imp.head(5).to_string(index=False)}")

In [ ]:
# ── CELL 19 : Stage 2 — Severity on predicted fault rows ─────────────────────
# PS1 = binary will_fail_7d.
# Stage 2 = on rows predicted=1, classify SEVERITY (MINOR/MAJOR/CRITICAL).
# Source: failure_level from CTA.SERVICENOW_AVAILABILITY_EVENTS (3-class).
# Logged as a separate run under the same experiment.

if "failure_level" in df.columns:
    sev_train = train_df[train_df["will_fail_7d"] == 1].dropna(subset=["failure_level"])
    sev_test  = test_df[test_df["will_fail_7d"]  == 1].dropna(subset=["failure_level"])

    Xs_tr = sev_train[FEATURE_COLS].values
    ys_tr = sev_train["failure_level"].values.astype(int) - 1   # 0/1/2
    Xs_te = sev_test[FEATURE_COLS].values
    ys_te = sev_test["failure_level"].values.astype(int) - 1

    with mlflow.start_run(run_name="stage2_severity") as run:
        mlflow.set_tags({**COMMON_TAGS,
                         "model_type":"xgboost_multiclass",
                         "stage":"2", "num_classes":"3"})

        sev_model = XGBClassifier(
            n_estimators=300, max_depth=5, learning_rate=0.05,
            objective="multi:softprob", num_class=3,
            tree_method="hist", random_state=42, n_jobs=-1,
        )
        sev_model.fit(Xs_tr, ys_tr, verbose=False)

        sev_prob = sev_model.predict_proba(Xs_te)
        sev_pred = sev_model.predict(Xs_te)
        sev_auc  = roc_auc_score(ys_te, sev_prob, multi_class="ovr", average="macro")
        sev_f1   = f1_score(ys_te, sev_pred, average="macro")

        mlflow.log_metrics({"test_severity_auc": round(sev_auc, 4),
                             "test_severity_f1" : round(sev_f1, 4),
                             "stage2_train_n"   : len(Xs_tr),
                             "stage2_test_n"    : len(Xs_te)})

        sig = infer_signature(Xs_tr, sev_model.predict_proba(Xs_tr))
        mlflow.sklearn.log_model(sev_model, "severity_model", signature=sig)
        SEV_RUN = run.info.run_id

    print(f"\nStage 2 Severity  AUC(macro)={sev_auc:.4f}  F1(macro)={sev_f1:.4f}")
    print(classification_report(ys_te, sev_pred,
                                 target_names=["MINOR","MAJOR","CRITICAL"]))
else:
    print("failure_level not in Gold — Stage 2 skipped.")
    sev_model = None

In [ ]:
# ── CELL 20 : Select champion model ──────────────────────────────────────────
all_runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    filter_string="tags.stage != '2'",
    order_by=["metrics.test_auc DESC"],
)
keep_cols = [c for c in
             ["tags.mlflow.runName","metrics.test_auc","metrics.test_f1",
              "metrics.test_ap","metrics.val_auc","run_id"]
             if c in all_runs.columns]
print("\n── All runs ranked by test AUC ─────────────────────────────────")
print(all_runs[keep_cols].head(10).to_string(index=False))

CHAMPION_RUN_ID = all_runs.iloc[0]["run_id"]
CHAMPION_NAME   = all_runs.iloc[0]["tags.mlflow.runName"]
CHAMPION_AUC    = all_runs.iloc[0]["metrics.test_auc"]
print(f"\n★  Champion: {CHAMPION_NAME}  AUC={CHAMPION_AUC:.4f}  run={CHAMPION_RUN_ID[:8]}…")

In [ ]:
# ── CELL 21 : Register champion in MLflow Model Registry ─────────────────────
# Staging  → QA / integration tests
# Production → live inference (promote after QA passes)
# Archived → superseded versions (auto-archived when new Production is promoted)

mlfc = mlflow.tracking.MlflowClient()

try:
    mlfc.create_registered_model(
        name=MLFLOW_MODEL_NAME,
        description=(
            "PS1 Device Failure Prediction — Chicago Ventra CTA. "
            "Binary: will_fail_7d. Grain: device×day. "
            "Architecture: CUBIC MARS AWS v6."
        ),
        tags={"ps":"PS1","target":"will_fail_7d","city":"chicago"},
    )
    print(f"Created MLflow registered model: {MLFLOW_MODEL_NAME}")
except mlflow.exceptions.MlflowException:
    print(f"Model already exists: {MLFLOW_MODEL_NAME}")

mv = mlfc.create_model_version(
    name        = MLFLOW_MODEL_NAME,
    source      = f"runs:/{CHAMPION_RUN_ID}/model",
    run_id      = CHAMPION_RUN_ID,
    description = (
        f"Champion: {CHAMPION_NAME}  AUC={CHAMPION_AUC:.4f}  "
        f"Gold v{GOLD_VERSION}  Train: 2024-01-01→2025-09-30"
    ),
    tags={"algorithm":CHAMPION_NAME, "test_auc":str(CHAMPION_AUC),
          "gold_version":str(GOLD_VERSION), "n_features":str(len(FEATURE_COLS))},
)
mlfc.transition_model_version_stage(
    name=MLFLOW_MODEL_NAME, version=mv.version,
    stage="Staging", archive_existing_versions=False)

MLFLOW_VERSION = mv.version
print(f"MLflow Model: {MLFLOW_MODEL_NAME}  v{MLFLOW_VERSION}  → Staging")
print("To promote to Production after QA:")
print(f"  mlfc.transition_model_version_stage(name='{MLFLOW_MODEL_NAME}',")
print(f"    version='{MLFLOW_VERSION}', stage='Production',")
print(f"    archive_existing_versions=True)")

In [ ]:
# ── CELL 22 : Register in SageMaker Model Registry ───────────────────────────
# MLflow Registry  = experiment & version lineage (developer-facing)
# SageMaker Registry = deployment gate — PendingManualApproval → Approved → Endpoint
# Both are maintained in parallel per CUBIC MARS AWS v6 architecture.

# Load champion from MLflow and serialise to model.tar.gz
champion_model = mlflow.sklearn.load_model(f"runs:/{CHAMPION_RUN_ID}/model")
joblib.dump(champion_model,  "/tmp/ps1_champion.joblib")
joblib.dump(FEATURE_COLS,    "/tmp/ps1_feature_cols.joblib")

test_metrics = evaluate(champion_model, X_test, y_test, "test")
S3_CLIENT.put_object(
    Bucket=ARTIFACT_BUCKET, Key="sagemaker/ps1/metrics/test_metrics.json",
    Body=json.dumps(test_metrics), ContentType="application/json")

with tarfile.open("/tmp/model.tar.gz", "w:gz") as tar:
    tar.add("/tmp/ps1_champion.joblib",    arcname="ps1_champion.joblib")
    tar.add("/tmp/ps1_feature_cols.joblib",arcname="ps1_feature_cols.joblib")

MODEL_S3_KEY = f"sagemaker/ps1/model-v{MLFLOW_VERSION}/model.tar.gz"
S3_CLIENT.upload_file("/tmp/model.tar.gz", ARTIFACT_BUCKET, MODEL_S3_KEY)
MODEL_DATA_URL = f"s3://{ARTIFACT_BUCKET}/{MODEL_S3_KEY}"
print(f"Artifact: {MODEL_DATA_URL}")

import sagemaker.image_uris as sm_images
image_uri = sm_images.retrieve(
    framework="sklearn", region=REGION, version="1.2-1", instance_type="ml.m5.large", image_scope="inference")

# Create Model Package Group (once per PS — idempotent)
try:
    SM_CLIENT.create_model_package_group(
        ModelPackageGroupName=MODEL_REGISTRY_NAME,
        ModelPackageGroupDescription=(
            "PS1 Device Failure Prediction — Chicago Ventra. "
            "Binary: will_fail_7d. CUBIC MARS AWS v6."),
    )
except SM_CLIENT.exceptions.ClientError:
    pass   # already exists

resp = SM_CLIENT.create_model_package(
    ModelPackageGroupName = MODEL_REGISTRY_NAME,
    ModelPackageDescription = (
        f"{CHAMPION_NAME} | AUC={CHAMPION_AUC:.4f} | "
        f"Gold v{GOLD_VERSION} | MLflow v{MLFLOW_VERSION}"),
    InferenceSpecification = {
        "Containers": [{"Image": image_uri, "ModelDataUrl": MODEL_DATA_URL}],
        "SupportedContentTypes"                 : ["text/csv","application/json"],
        "SupportedResponseMIMETypes"             : ["application/json"],
        "SupportedTransformInstanceTypes"        : ["ml.m5.large","ml.m5.xlarge"],
        "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.large","ml.m5.xlarge"],
    },
    ModelApprovalStatus = "PendingManualApproval",
    ModelMetrics={
        "ModelQuality": {"Statistics": {
            "ContentType": "application/json",
            "S3Uri": f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1/metrics/test_metrics.json",
        }}
    },
    CustomerMetadataProperties={
        "ps":"PS1", "target":"will_fail_7d", "city":"chicago",
        "algorithm":CHAMPION_NAME, "test_auc":str(round(CHAMPION_AUC,4)),
        "gold_version":str(GOLD_VERSION), "mlflow_version":str(MLFLOW_VERSION),
        "mlflow_run_id":CHAMPION_RUN_ID, "n_features":str(len(FEATURE_COLS)),
    },
)
SM_MODEL_PKG_ARN = resp["ModelPackageArn"]
print(f"SageMaker Package: {SM_MODEL_PKG_ARN}")
print("Status: PendingManualApproval")
print("→ Studio: Models → Model Registry → Approve, OR run Cell 23 to approve + deploy")

In [ ]:
# ── CELL 23 : Approve + deploy to SageMaker Endpoint ────────────────────────
# Run AFTER reviewing test metrics.
# DataCaptureConfig records every inference input + output → used by Model Monitor.

SM_CLIENT.update_model_package(
    ModelPackageArn=SM_MODEL_PKG_ARN,
    ModelApprovalStatus="Approved")
print("Approved.")

from sagemaker.model import ModelPackage
from sagemaker.model_monitor import DataCaptureConfig

registered = ModelPackage(
    role=ROLE, model_package_arn=SM_MODEL_PKG_ARN,
    sagemaker_session=SM_SESSION)

predictor = registered.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=ENDPOINT_NAME,
    data_capture_config=DataCaptureConfig(
        enable_capture       = True,
        sampling_percentage  = 100,
        destination_s3_uri   = f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1/data-capture/",
        capture_options      = ["Input","Output"],
    ),
)
print(f"Endpoint live: {ENDPOINT_NAME}")

# Quick smoke test
sample_csv = pd.DataFrame(X_test[:3], columns=FEATURE_COLS).to_csv(
    index=False, header=False)
result = predictor.predict(sample_csv,
                           initial_args={"ContentType":"text/csv"})
print(f"Sample predictions: {result}")

In [ ]:
# ── CELL 24 : SageMaker Model Monitor — Data Quality Baseline ────────────────
# Baseline = statistics + constraints computed from TRAINING data.
# Daily monitor compares live endpoint inputs to baseline.
# Violations → S3 report + CloudWatch alarm (97% quality gate = 3% violation threshold).

from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat
from sagemaker.model_monitor import CronExpressionGenerator

monitor = DefaultModelMonitor(
    role=ROLE, instance_count=1, instance_type="ml.m5.large",
    volume_size_in_gb=20, max_runtime_in_seconds=3600,
    sagemaker_session=SM_SESSION,
)

# Upload baseline data (training features)
baseline_df = pd.DataFrame(X_train, columns=FEATURE_COLS)
baseline_df.to_csv("/tmp/baseline.csv", index=False)
S3_CLIENT.upload_file("/tmp/baseline.csv", ARTIFACT_BUCKET,
                      "sagemaker/ps1/baseline/baseline.csv")
BASELINE_S3 = f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1/baseline/baseline.csv"

monitor.suggest_baseline(
    baseline_dataset = BASELINE_S3,
    dataset_format   = DatasetFormat.csv(header=True),
    output_s3_uri    = f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1/baseline/output/",
    wait=True, logs=True,
)
print("Baseline job complete.")
print(f"Constraints : s3://{ARTIFACT_BUCKET}/sagemaker/ps1/baseline/output/constraints.json")
print(f"Statistics  : s3://{ARTIFACT_BUCKET}/sagemaker/ps1/baseline/output/statistics.json")

In [ ]:
# ── CELL 25 : Schedule daily Data Quality monitoring ─────────────────────────
monitor.create_monitoring_schedule(
    monitor_schedule_name    = "chicago-ps1-data-quality",
    endpoint_input           = ENDPOINT_NAME,
    output_s3_uri            = f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1/monitor-output/",
    statistics               = monitor.baseline_statistics(),
    constraints              = monitor.suggested_constraints(),
    schedule_cron_expression = CronExpressionGenerator.daily(),
    enable_cloudwatch_metrics= True,
)
print("Data Quality Monitor scheduled → daily 00:00 UTC")

# CloudWatch alarm: >3% feature violations = 97% quality gate breach
cw = boto3.client("cloudwatch", region_name=REGION)
cw.put_metric_alarm(
    AlarmName          = "PS1-DataQuality-ViolationRate",
    ComparisonOperator = "GreaterThanThreshold",
    EvaluationPeriods  = 1,
    MetricName         = "feature_baseline_drift_percentage",
    Namespace          = "aws/sagemaker/Endpoints/data-metrics",
    Period             = 86400,
    Statistic          = "Average",
    Threshold          = 3.0,    # 97% quality gate → alarm on >3% violations
    AlarmDescription   = "PS1 input feature violation rate > 3% (CUBIC MARS AWS v6 97% quality gate)",
    Dimensions         = [{"Name":"Endpoint","Value":ENDPOINT_NAME}],
    TreatMissingData   = "notBreaching",
)
print("CloudWatch alarm set: PS1-DataQuality-ViolationRate (threshold=3%)")

In [ ]:
# ── CELL 26 : Model Quality Monitor — track AUC drift over time ──────────────
# Compares live predictions to ground-truth labels (uploaded after 7-day window).
# Ground truth path: s3://ARTIFACT_BUCKET/sagemaker/ps1/ground-truth/
# Upload actual outcomes daily after they resolve (event_date + 7 days).

from sagemaker.model_monitor import ModelQualityMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat

mq = ModelQualityMonitor(
    role=ROLE, instance_count=1, instance_type="ml.m5.large",
    sagemaker_session=SM_SESSION,
)

# Baseline: test set predictions vs actuals
y_prob_test = champion_model.predict_proba(X_test)[:,1]
mq_baseline = pd.DataFrame({
    "prediction" : y_prob_test,
    "probability": y_prob_test,
    "label"      : y_test.astype(str),
})
mq_baseline.to_csv("/tmp/mq_baseline.csv", index=False)
S3_CLIENT.upload_file("/tmp/mq_baseline.csv", ARTIFACT_BUCKET,
                      "sagemaker/ps1/mq-baseline/mq_baseline.csv")

mq.suggest_baseline(
    baseline_dataset = f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1/mq-baseline/mq_baseline.csv",
    dataset_format   = DatasetFormat.csv(header=True),
    output_s3_uri    = f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1/mq-baseline/output/",
    problem_type     = "BinaryClassification",
    inference_attribute             = "prediction",
    probability_attribute           = "probability",
    ground_truth_attribute          = "label",
    probability_threshold_attribute = "0.5",
    wait=True,
)

mq.create_monitoring_schedule(
    monitor_schedule_name    = "chicago-ps1-model-quality",
    endpoint_input           = ENDPOINT_NAME,
    ground_truth_input       = f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1/ground-truth/",
    problem_type             = "BinaryClassification",
    output_s3_uri            = f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1/mq-output/",
    constraints              = mq.suggested_constraints(),
    schedule_cron_expression = CronExpressionGenerator.daily(),
    enable_cloudwatch_metrics= True,
)
print("Model Quality Monitor scheduled → chicago-ps1-model-quality")
print(f"Ground truth upload path:")
print(f"  s3://{ARTIFACT_BUCKET}/sagemaker/ps1/ground-truth/")
print("Upload: YYYY/MM/DD/HH/groundtruth_YYYYMMDD.jsonl  (one label per inference record)")

In [ ]:
# ── CELL 27 : Final summary ───────────────────────────────────────────────────
print("=" * 65)
print("PS1 — Complete  (CUBIC MARS AWS v6)")
print("=" * 65)
print(f"\n{'Item':<28} {'Value'}")
print("-" * 65)
items = [
    ("MLflow Experiment",   EXPERIMENT_NAME),
    ("MLflow Model",        f"{MLFLOW_MODEL_NAME}  v{MLFLOW_VERSION}  [Staging]"),
    ("SM Model Registry",   MODEL_REGISTRY_NAME),
    ("SM Model Package",    SM_MODEL_PKG_ARN[-40:] + "…"),
    ("SM Endpoint",         ENDPOINT_NAME),
    ("Feature Store",       FS_GROUP_NAME),
    ("Data Quality Monitor","chicago-ps1-data-quality  (daily)"),
    ("Model Quality Monitor","chicago-ps1-model-quality  (daily)"),
    ("CW Alarm",            "PS1-DataQuality-ViolationRate  (>3%)"),
    ("Champion model",      f"{CHAMPION_NAME}  AUC={CHAMPION_AUC:.4f}"),
    ("Features",            str(len(FEATURE_COLS))),
    ("Gold Delta version",  str(GOLD_VERSION)),
    ("Artifacts bucket",    f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1/"),
]
for k, v in items:
    print(f"  {k:<26} {v}")

print("""
Retraining flow (next run):
  1. Re-run from Cell 4 — reads latest Gold Delta version
  2. MLflow auto-increments version number
  3. SM Registry adds new Package → PendingManualApproval
  4. Approve in Studio UI → new Endpoint deployment
  5. Previous Production version auto-archived in MLflow

For PS2–PS5: same notebook structure.
  Change: GOLD_PREFIX, TARGET, FEATURE_COLS, model hyperparams only.
""")